In [ ]:
import sys
sys.path.insert(0, '..')
from src.search.engine import LegalSemanticSearchEngine
from src.loader import load_json

In [ ]:
# Загружаем статьи
raw_D = load_json("/home/gshjis/Python_projects/IA_NPA_auditor/data/processed/const.json")
articles = [r["content"] for r in raw_D]

print(f"Загружено статей: {len(articles)}")


In [ ]:
tagger = LegalSemanticSearchEngine(
    tags_filepath="/home/gshjis/Python_projects/IA_NPA_auditor/data/raw/base.json", 
    training_corpus=articles,
    tags_per_article=20
)

In [ ]:
# ============================================================================
# ТЕСТ 1: Проверка тегов для проблемного запроса
# ============================================================================
art_44 = ""
for i in raw_D:
    if i["number"] == 44:
        art_44 = i["content"]
        break

tests = [t["content"] for t in raw_D[:5]]
tests.append("'Как наследуется квартира?'")
tests.append(art_44)
for query in tests:
    tags = tagger.get_tag_recommendations_formatted(query)
    print(tags)



In [ ]:
diverse_queries = [
    # ===== ПРАВА ЧЕЛОВЕКА (5 запросов) =====
    "Какие личные права и свободы гарантированы гражданам?",
    "В каких случаях допускается ограничение прав и свобод?",
    "Какие существуют политические права граждан?",
    "Что такое право на неприкосновенность частной жизни?",
    "Какие гарантии равенства перед законом предусмотрены?",

    # ===== СЕМЕЙНОЕ ПРАВО (5 запросов) =====
    "Каковы права и обязанности родителей по воспитанию детей?",
    "Что говорит конституция о защите материнства и детства?",
    "Какие права имеют дети в семье?",
    "Как государство защищает семью?",
    "Каковы обязанности детей по отношению к родителям?",

    # ===== СОБСТВЕННОСТЬ И НАСЛЕДОВАНИЕ (5 запросов) =====
    "Какие существуют формы собственности?",
    "Как государство защищает право собственности?",
    "Что такое право наследования и как оно охраняется?",
    "В каких случаях возможно принудительное изъятие имущества?",
    "Какие гарантии защиты сбережений граждан существуют?",

    # ===== ТРУД И СОЦИАЛЬНОЕ ОБЕСПЕЧЕНИЕ (5 запросов) =====
    "Какие трудовые права гарантированы гражданам?",
    "Что такое право на справедливое вознаграждение за труд?",
    "Какие гарантии социального обеспечения предусмотрены?",
    "Что говорит конституция о праве на отдых?",
    "Какие гарантии имеют пенсионеры и инвалиды?",

    # ===== ЖИЛИЩНЫЕ ПРАВА (3 запроса) =====
    "Что такое право на жилище и как оно обеспечивается?",
    "Кто имеет право на бесплатное жилье от государства?",
    "Какие гарантии неприкосновенности жилища существуют?",

    # ===== ОБРАЗОВАНИЕ И КУЛЬТУРА (3 запроса) =====
    "Какие гарантии права на образование предусмотрены?",
    "Что такое доступность образования?",
    "Какие права в сфере культуры и творчества гарантированы?",

    # ===== ЗДРАВООХРАНЕНИЕ (3 запросы) =====
    "Какие гарантии права на охрану здоровья существуют?",
    "Что такое бесплатное медицинское обслуживание?",
    "Как государство заботится о здоровье граждан?",

    # ===== ОРГАНЫ ВЛАСТИ (5 запросов) =====
    "Кто может быть избран президентом и каковы его полномочия?",
    "Как формируется парламент и каковы его функции?",
    "Что такое правительство и чем оно занимается?",
    "Какова роль Конституционного Суда?",
    "Что такое местное самоуправление и как оно организовано?",

    # ===== ИЗБИРАТЕЛЬНАЯ СИСТЕМА (4 запроса) =====
    "Кто имеет право избирать и быть избранным?",
    "Какие принципы избирательного права существуют?",
    "Что такое референдум и как он проводится?",
    "Как обеспечивается тайна голосования?",

    # ===== МЕЖДУНАРОДНЫЕ ОТНОШЕНИЯ (3 запроса) =====
    "Как соотносятся международное право и национальное законодательство?",
    "Каковы принципы внешней политики?",
    "Что такое право убежища и кому оно предоставляется?",

    # ===== ГРАЖДАНСТВО (3 запроса) =====
    "Кто является гражданином Республики Беларусь?",
    "Можно ли лишить гражданства?",
    "Каковы права иностранцев и лиц без гражданства?",

    # ===== ОБЯЗАННОСТИ (3 запроса) =====
    "Какие конституционные обязанности есть у граждан?",
    "Что такое воинская обязанность?",
    "Каковы обязанности по охране природы?",

    # ===== ЧРЕЗВЫЧАЙНЫЕ СИТУАЦИИ (3 запроса) =====
    "Что такое чрезвычайное положение?",
    "Какие права могут быть ограничены в условиях чрезвычайного положения?",
    "Что такое военное положение?",
]

print(f"Всего запросов: {len(diverse_queries)}")

In [ ]:
for idx, query in enumerate(diverse_queries, 1):
    print("\n" + "=" * 80)
    print(f"📌 ЗАПРОС №{idx}".center(80))
    print("=" * 80)
    print(f"{query}\n")
    
    results = tagger.find_articles_by_new_sentence(query, k=10)
    
    for rank, res in enumerate(results, 1):
        # Определяем иконку по score
        if res['score'] >= 0.6:
            icon = "🟢🔝"
        elif res['score'] >= 0.5:
            icon = "🟡"
        else:
            icon = "⚪"
        
        # Основная информация
        print(f"{icon} [{rank}] Score: {res['score']:.3f}")
        
        # Текст статьи (первые 150 символов)
        article_text = res['article']['content'] if isinstance(res['article'], dict) else str(res['article'])
        print(f"   📄 {article_text[:150]}...")
        
        # ТЕГИ: топ-5 тегов запроса
        print(f"   🔍 Теги запроса (топ-5):", end=" ")
        query_tags_top = sorted(res['query_tags'].items(), key=lambda x: x[1], reverse=True)[:5]
        for tag, weight in query_tags_top:
            print(f"{tag}({weight:.2f})", end=" ")
        print()
        
        # ТЕГИ: топ-5 общих тегов
        print(f"   🤝 Общие теги ({len(res['common_tags'])}):", end=" ")
        for tag in res['common_tags'][:5]:
            # берем вес из article_tags для этих тегов
            if tag in res['article_tags']:
                print(f"{tag}({res['article_tags'][tag]:.2f})", end=" ")
        print()
        
        # ТЕГИ: топ-3 уникальных тега статьи (не в запросе)
        article_unique = {tag: weight for tag, weight in res['article_tags'].items() 
                         if tag not in res['query_tags']}
        if article_unique:
            print(f"   🏷️ Уникальные теги статьи (топ-3):", end=" ")
            for tag, weight in sorted(article_unique.items(), key=lambda x: x[1], reverse=True)[:3]:
                print(f"{tag}({weight:.2f})", end=" ")
            print()
        
        print()
    
    print("-" * 80)

In [ ]:
res